In [1]:
import os, glob, json, base64
import torch
import torch.nn as nn
import networkx as nx
from sentence_transformers import SentenceTransformer

def process_kaggle_dataset(dataset_name="synthetic-ehr-synthea-1200-patients"):
    base_path = f"/kaggle/input/{dataset_name}"
    # Adjust path if your zip contained a nested folder (e.g., /fhir)
    file_paths = glob.glob(os.path.join(base_path, "**/*.json"), recursive=True)
    
    encounters = {}
    for path in file_paths:
        if os.path.basename(path) in ["organizations.json", "practitioners.json"]:
            continue
        with open(path, 'r') as f:
            bundle = json.load(f)
            
        for entry in bundle.get("entry", []):
            res = entry.get("resource", {})
            if res.get("resourceType") == "Encounter":
                encounters[res["id"]] = {"nodes": [], "note_text": "", "target": 0}
                
        for entry in bundle.get("entry", []):
            res = entry.get("resource", {})
            enc_ref = res.get("encounter", {}).get("reference", "").split(":")[-1]
            
            if enc_ref in encounters:
                if res.get("resourceType") in ["Condition", "Observation", "Procedure"]:
                    code = res.get("code", {}).get("coding", [{}])[0].get("code")
                    if code:
                        encounters[enc_ref]["nodes"].append(code)
                    if code in ["230690007", "19169002"]: # Stroke / Miscarriage
                        encounters[enc_ref]["target"] = 1
                elif res.get("resourceType") == "DiagnosticReport" and "presentedForm" in res:
                    raw_b64 = res["presentedForm"][0].get("data", "")
                    encounters[enc_ref]["note_text"] += " " + base64.b64decode(raw_b64).decode("utf-8", errors="ignore")
                    
    return {k: v for k, v in encounters.items() if v["nodes"] or v["note_text"]}

dataset = process_kaggle_dataset()

In [2]:
def generate_embeddings(dataset):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Extract unique nodes and texts
    unique_codes = list(set(code for enc in dataset.values() for code in enc["nodes"]))
    note_texts = [enc["note_text"] for enc in dataset.values()]
    
    # Semantic Embeddings (C_v and N_e)
    C_v = torch.tensor(model.encode(unique_codes))
    N_e = torch.tensor(model.encode(note_texts))
    
    # Structural Embeddings (S_v) via bipartite graph projection
    G = nx.Graph()
    for enc_id, data in dataset.items():
        for code in data["nodes"]:
            G.add_edge(enc_id, code)
    
    # Placeholder for DeepWalk implementation
    S_v = torch.randn(len(unique_codes), 64) 
    
    # Combined Node Initialization
    X_v = torch.cat([S_v, C_v], dim=-1)
    
    # Level 2: Hyperedge Semantics H_e
    mlp = nn.Sequential(nn.Linear(N_e.shape[1] + C_v.shape[1], 128), nn.ReLU())
    # Simplified pooling of C_v for demonstration
    pooled_C_v = C_v.mean(dim=0).unsqueeze(0).repeat(N_e.shape[0], 1)
    H_e = mlp(torch.cat([N_e, pooled_C_v], dim=-1))
    
    return X_v, H_e

In [3]:
class MINGLELayer(nn.Module):
    def __init__(self, node_dim, edge_dim, sem_dim):
        super().__init__()
        self.v2e_proj = nn.Linear(node_dim, edge_dim)
        self.mlp_edge = nn.Sequential(nn.Linear(edge_dim + sem_dim, edge_dim), nn.ReLU())
        self.e2v_proj = nn.Linear(edge_dim, node_dim)
        self.mlp_node = nn.Sequential(nn.Linear(node_dim, node_dim), nn.ReLU())

    def forward(self, X_v, E_e, H_e, incidence_matrix):
        # Nodes to Hyperedges
        v2e = torch.matmul(incidence_matrix.t(), self.v2e_proj(X_v))
        E_next = self.mlp_edge(torch.cat([v2e, H_e], dim=-1))
        
        # Hyperedges to Nodes
        e2v = torch.matmul(incidence_matrix, self.e2v_proj(E_next))
        X_next = self.mlp_node(e2v)
        
        return X_next, E_next

In [4]:
import os, json, base64
import torch
import torch.nn as nn
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# 1. Auto-Discover and Parse Dataset
# ---------------------------------------------------------
def process_kaggle_dataset(base_path="/kaggle/input"):
    all_json_files = []
    
    # Auto-hunt for JSON files across all Kaggle input subdirectories
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.endswith(".json") and file not in ["organizations.json", "practitioners.json"]:
                all_json_files.append(os.path.join(root, file))
                
    print(f" Found {len(all_json_files)} patient JSON files.")
    if len(all_json_files) == 0:
        raise ValueError("No JSON files found! Double check your Kaggle Dataset name/attachment.")

    encounters = {}
    
    # First pass: Initialize encounters
    for path in all_json_files:
        with open(path, 'r') as f:
            try:
                bundle = json.load(f)
            except json.JSONDecodeError:
                continue
                
            for entry in bundle.get("entry", []):
                res = entry.get("resource", {})
                if res.get("resourceType") == "Encounter":
                    encounters[res["id"]] = {"nodes": [], "note_text": "", "target": 0}
                    
            # Second pass: Map nodes and text
            for entry in bundle.get("entry", []):
                res = entry.get("resource", {})
                enc_ref = res.get("encounter", {}).get("reference", "").split(":")[-1]
                
                if enc_ref in encounters:
                    # Extract codes
                    if res.get("resourceType") in ["Condition", "Observation", "Procedure"]:
                        code = res.get("code", {}).get("coding", [{}])[0].get("code")
                        if code:
                            encounters[enc_ref]["nodes"].append(code)
                        # Target: Stroke (230690007) or Miscarriage (19169002)
                        if code in ["230690007", "19169002"]: 
                            encounters[enc_ref]["target"] = 1
                            
                    # Extract notes
                    elif res.get("resourceType") == "DiagnosticReport" and "presentedForm" in res:
                        raw_b64 = res["presentedForm"][0].get("data", "")
                        if raw_b64:
                            decoded = base64.b64decode(raw_b64).decode("utf-8", errors="ignore")
                            encounters[enc_ref]["note_text"] += " " + decoded.strip()
                            
    # Keep only populated encounters
    return {k: v for k, v in encounters.items() if v["nodes"] or v["note_text"]}

dataset = process_kaggle_dataset()
print(f" Extracted {len(dataset)} valid encounters.")

# ---------------------------------------------------------
# 2. Safe Embedding Generation
# ---------------------------------------------------------
def generate_embeddings(dataset):
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    # Extract unique nodes and texts
    unique_codes = list(set(code for enc in dataset.values() for code in enc["nodes"]))
    note_texts = [enc["note_text"] if enc["note_text"].strip() != "" else "No clinical notes" for enc in dataset.values()]
    
    print(f" Encoding {len(unique_codes)} unique medical codes and {len(note_texts)} clinical notes...")
    
    # Semantic Embeddings
    C_v = torch.tensor(model.encode(unique_codes))
    N_e = torch.tensor(model.encode(note_texts))
    
    # Ensure 2D shapes (Safety Check)
    if C_v.dim() == 1: C_v = C_v.unsqueeze(0)
    if N_e.dim() == 1: N_e = N_e.unsqueeze(0)
    
    # Placeholder Structural Embeddings (DeepWalk substitute for now)
    S_v = torch.randn(len(unique_codes), 64) 
    
    # Level 1: Combined Node Initialization
    X_v = torch.cat([S_v, C_v], dim=-1)
    
    # Level 2: Hyperedge Semantics H_e
    mlp = nn.Sequential(nn.Linear(N_e.shape[1] + C_v.shape[1], 128), nn.ReLU())
    
    # Pool C_v safely across dimension 0
    pooled_C_v = C_v.mean(dim=0).unsqueeze(0).repeat(N_e.shape[0], 1)
    H_e = mlp(torch.cat([N_e, pooled_C_v], dim=-1))
    
    return X_v, H_e, len(unique_codes)

X_v, H_e, num_unique_nodes = generate_embeddings(dataset)
print(f"Embeddings generated! X_v shape: {X_v.shape}, H_e shape: {H_e.shape}")

# ---------------------------------------------------------
# 3. Initialize Variables for Network Pass
# ---------------------------------------------------------
num_edges = len(dataset)
# Placeholder for Incidence Matrix (Nodes x Edges)
incidence_matrix = torch.zeros(num_unique_nodes, num_edges)

# Initial empty hyperedge representations
E_e = torch.zeros(num_edges, 128)

print("Ready for the MINGLE Layer forward pass!")

 Found 1278 patient JSON files.
 Extracted 143946 valid encounters.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Encoding 441 unique medical codes and 143946 clinical notes...
Embeddings generated! X_v shape: torch.Size([441, 448]), H_e shape: torch.Size([143946, 128])
Ready for the MINGLE Layer forward pass!


In [5]:
import torch
import torch.nn as nn

# ---------------------------------------------------------
# 1. Build the Incidence Matrix
# ---------------------------------------------------------
print("Constructing Incidence Matrix...")

# Extract stable lists to maintain indexing order
unique_codes_list = list(set(code for enc in dataset.values() for code in enc["nodes"]))
encounter_ids_list = list(dataset.keys())

# Create lookup dictionaries for fast indexing
code_to_idx = {code: i for i, code in enumerate(unique_codes_list)}
enc_to_idx = {enc_id: i for i, enc_id in enumerate(encounter_ids_list)}

num_nodes = len(unique_codes_list)
num_edges = len(encounter_ids_list)

# Initialize the Incidence Matrix: Shape [441, 143946]
incidence_matrix = torch.zeros(num_nodes, num_edges)

# Populate the matrix: 1.0 if a code exists in an encounter
for enc_id, data in dataset.items():
    e_idx = enc_to_idx[enc_id]
    for code in data["nodes"]:
        v_idx = code_to_idx[code]
        incidence_matrix[v_idx, e_idx] = 1.0

print(f"Incidence Matrix populated with shape: {incidence_matrix.shape}")

# ---------------------------------------------------------
# 2. Define the MINGLE Hypergraph Layer
# ---------------------------------------------------------
class MINGLELayer(nn.Module):
    def __init__(self, node_dim, edge_dim, sem_dim):
        super().__init__()
        # Projections for message passing
        self.v2e_proj = nn.Linear(node_dim, edge_dim)
        self.e2v_proj = nn.Linear(edge_dim, node_dim)
        
        # Eq 7: Hyperedge update fusing aggregated nodes with note semantics H_e
        self.mlp_edge = nn.Sequential(
            nn.Linear(edge_dim + sem_dim, edge_dim),
            nn.ReLU()
        )
        # Eq 1: Node update
        self.mlp_node = nn.Sequential(
            nn.Linear(node_dim, node_dim),
            nn.ReLU()
        )

    def forward(self, X_v, E_e, H_e, inc_mat):
        # Calculate degrees to normalize message passing (prevent exploding values)
        deg_e = inc_mat.sum(dim=0, keepdim=True).clamp(min=1.0) # [1, num_edges]
        deg_v = inc_mat.sum(dim=1, keepdim=True).clamp(min=1.0) # [num_nodes, 1]
        
        # --- f_{V -> E}: Node to Hyperedge Passing ---
        proj_nodes = self.v2e_proj(X_v) 
        # Matrix multiply transposes the incidence matrix to aggregate nodes into edges
        v2e_agg = torch.matmul(inc_mat.t(), proj_nodes) / deg_e.t() 
        
        # Fuse with Hyperedge Semantics (H_e)
        E_next = self.mlp_edge(torch.cat([v2e_agg, H_e], dim=-1))
        
        # --- f_{E -> V}: Hyperedge to Node Passing ---
        proj_edges = self.e2v_proj(E_next) 
        e2v_agg = torch.matmul(inc_mat, proj_edges) / deg_v
        X_next = self.mlp_node(e2v_agg)
        
        return X_next, E_next

# ---------------------------------------------------------
# 3. Execute the Forward Pass
# ---------------------------------------------------------
# node_dim=448 (from your X_v), edge_dim=128 (hidden size), sem_dim=128 (from your H_e)
layer = MINGLELayer(node_dim=448, edge_dim=128, sem_dim=128)

# Initialize empty hyperedge embeddings for the start of layer 1
E_e_init = torch.zeros(num_edges, 128)

print("Executing MINGLE Layer 1 forward pass...")
# Note: In a real training loop, you would call detach().numpy() only when pulling to CPU for metrics
X_v_updated, E_e_updated = layer(X_v, E_e_init, H_e, incidence_matrix)

print(f"Success! Updated Node Tensor Shape: {X_v_updated.shape}")
print(f"Success! Updated Hyperedge Tensor Shape: {E_e_updated.shape}")

Constructing Incidence Matrix...
Incidence Matrix populated with shape: torch.Size([441, 143946])
Executing MINGLE Layer 1 forward pass...
Success! Updated Node Tensor Shape: torch.Size([441, 448])
Success! Updated Hyperedge Tensor Shape: torch.Size([143946, 128])


In [6]:
import torch.optim as optim
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score

# ---------------------------------------------------------
# 1. Define the Full Multi-Layer Classifier
# ---------------------------------------------------------
class MINGLEClassifier(nn.Module):
    def __init__(self, node_dim, edge_dim, sem_dim, num_layers=2):
        super().__init__()
        self.layers = nn.ModuleList([
            MINGLELayer(node_dim, edge_dim, sem_dim) for _ in range(num_layers)
        ])
        
        # Eq. 3: Classifier over concatenated layer outputs
        self.classifier = nn.Sequential(
            nn.Linear(num_layers * edge_dim, edge_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(edge_dim, 1) # Binary classification (e.g., Stroke/Miscarriage)
        )

    def forward(self, X_v, H_e, inc_mat):
        E_e = torch.zeros(inc_mat.shape[1], H_e.shape[1], device=X_v.device)
        edge_layer_embeddings = []

        for layer in self.layers:
            X_v, E_e = layer(X_v, E_e, H_e, inc_mat)
            edge_layer_embeddings.append(E_e)

        # Concatenate hyperedge embeddings across all layers
        multi_layer_edge_repr = torch.cat(edge_layer_embeddings, dim=-1)
        logits = self.classifier(multi_layer_edge_repr)
        return logits

# ---------------------------------------------------------
# 2. Setup Targets and Optimization (With Weighting)
# ---------------------------------------------------------
print("Setting up training loop with class weighting...")

y_true = torch.tensor([dataset[enc_id]["target"] for enc_id in encounter_ids_list], dtype=torch.float32)

# Calculate imbalance ratio to weight the rare positive class
num_positives = y_true.sum()
num_negatives = len(y_true) - num_positives
pos_weight_val = num_negatives / (num_positives + 1e-5) # Prevent division by zero
print(f"Dataset Split -> Negatives: {int(num_negatives)}, Positives: {int(num_positives)}")
print(f"Applying positive weight: {pos_weight_val:.2f}")

model = MINGLEClassifier(node_dim=448, edge_dim=128, sem_dim=128, num_layers=2)
# Apply the weight to the loss function
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val)) 
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

# ---------------------------------------------------------
# 3. Training Loop (Extended)
# ---------------------------------------------------------
print("Starting Training (Full Batch Graph)...")
epochs = 50 # Increased to allow network to learn minority features

X_v_input = X_v.detach()
H_e_input = H_e.detach()

model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    
    logits = model(X_v_input, H_e_input, incidence_matrix).squeeze()
    loss = criterion(logits, y_true)
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | Loss: {loss.item():.4f}")

# ---------------------------------------------------------
# 4. Evaluation Metrics
# ---------------------------------------------------------
model.eval()
with torch.no_grad():
    logits = model(X_v_input, H_e_input, incidence_matrix).squeeze()
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()
    
    # Safely move tensors to NumPy for sklearn metrics
    y_true_np = y_true.detach().numpy()
    probs_np = probs.detach().numpy()
    preds_np = preds.detach().numpy()

    # Calculate MINGLE Paper Metrics
    acc = accuracy_score(y_true_np, preds_np)
    auroc = roc_auc_score(y_true_np, probs_np)
    aupr = average_precision_score(y_true_np, probs_np)
    f1 = f1_score(y_true_np, preds_np, zero_division=0)

print("\n=== Final Model Metrics ===")
print(f"Accuracy : {acc * 100:.2f}%")
print(f"AUROC    : {auroc * 100:.2f}%")
print(f"AUPR     : {aupr * 100:.2f}%")
print(f"Macro-F1 : {f1 * 100:.2f}%")

Setting up training loop with class weighting...
Dataset Split -> Negatives: 142926, Positives: 1020
Applying positive weight: 140.12
Starting Training (Full Batch Graph)...


/tmp/ipykernel_23/3196617257.py:51: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val))


Epoch 05/50 | Loss: 1.3016
Epoch 10/50 | Loss: 1.0589
Epoch 15/50 | Loss: 0.6174
Epoch 20/50 | Loss: 0.3828
Epoch 25/50 | Loss: 0.2465
Epoch 30/50 | Loss: 0.1955
Epoch 35/50 | Loss: 0.1186
Epoch 40/50 | Loss: 0.0726
Epoch 45/50 | Loss: 0.0421
Epoch 50/50 | Loss: 0.0217

=== Final Model Metrics ===
Accuracy : 100.00%
AUROC    : 100.00%
AUPR     : 100.00%
Macro-F1 : 99.66%


In [7]:
import numpy as np
import torch
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score

# ---------------------------------------------------------
# 1. Stratified 7:1:2 Split Generation
# ---------------------------------------------------------
print("Creating Stratified Train/Val/Test Masks...")

# Convert tensor to numpy for scikit-learn splitting
y_true_np = y_true.detach().numpy()
indices = np.arange(num_edges)

# Split 1: 70% Train, 30% Temp (Val + Test)
idx_train, idx_temp, y_train, y_temp = train_test_split(
    indices, y_true_np, test_size=0.30, stratify=y_true_np, random_state=42
)

# Split 2: 1/3 of Temp into Val (10% total), 2/3 into Test (20% total)
idx_val, idx_test, _, _ = train_test_split(
    idx_temp, y_temp, test_size=0.6667, stratify=y_temp, random_state=42
)

# Initialize Boolean Masks
train_mask = torch.zeros(num_edges, dtype=torch.bool)
val_mask = torch.zeros(num_edges, dtype=torch.bool)
test_mask = torch.zeros(num_edges, dtype=torch.bool)

train_mask[idx_train] = True
val_mask[idx_val] = True
test_mask[idx_test] = True

print(f"Split sizes -> Train: {train_mask.sum()}, Val: {val_mask.sum()}, Test: {test_mask.sum()}")

# ---------------------------------------------------------
# 2. Re-initialize Model & Optimization
# ---------------------------------------------------------
# We must calculate the positive weight ONLY on the training set to prevent data leakage
num_train_pos = y_true[train_mask].sum()
num_train_neg = train_mask.sum() - num_train_pos
train_pos_weight = num_train_neg / (num_train_pos + 1e-5)

model = MINGLEClassifier(node_dim=448, edge_dim=128, sem_dim=128, num_layers=2)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(train_pos_weight))
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)

X_v_input = X_v.detach()
H_e_input = H_e.detach()

# ---------------------------------------------------------
# 3. Masked Training Loop
# ---------------------------------------------------------
print("\nStarting Training (Masked Graph)...")
epochs = 50

model.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # Forward pass over the entire graph structure
    logits = model(X_v_input, H_e_input, incidence_matrix).squeeze()
    
    # Apply Mask: Calculate loss ONLY on the training nodes
    loss = criterion(logits[train_mask], y_true[train_mask])
    
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{epochs} | Training Loss: {loss.item():.4f}")

# ---------------------------------------------------------
# 4. Unseen Test Set Evaluation
# ---------------------------------------------------------
model.eval()
with torch.no_grad():
    # Forward pass (model propagates features across the whole graph)
    logits = model(X_v_input, H_e_input, incidence_matrix).squeeze()
    
    # Apply Mask: Filter predictions to only the unseen Test set
    test_logits = logits[test_mask]
    test_y_true = y_true[test_mask]
    
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs > 0.5).float()
    
    # Extract to NumPy for sklearn metric calculations
    y_test_np = test_y_true.detach().numpy()
    probs_test_np = test_probs.detach().numpy()
    preds_test_np = test_preds.detach().numpy()

    # Calculate exact paper metrics
    acc = accuracy_score(y_test_np, preds_test_np)
    auroc = roc_auc_score(y_test_np, probs_test_np)
    aupr = average_precision_score(y_test_np, probs_test_np)
    f1 = f1_score(y_test_np, preds_test_np, zero_division=0)

print("\n=== Unseen Test Set Metrics (20% Holdout) ===")
print(f"Accuracy : {acc * 100:.2f}%")
print(f"AUROC    : {auroc * 100:.2f}%")
print(f"AUPR     : {aupr * 100:.2f}%")
print(f"Macro-F1 : {f1 * 100:.2f}%")

Creating Stratified Train/Val/Test Masks...
Split sizes -> Train: 100762, Val: 14393, Test: 28791

Starting Training (Masked Graph)...


/tmp/ipykernel_23/2961611545.py:46: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(train_pos_weight))


Epoch 05/50 | Training Loss: 1.3156
Epoch 10/50 | Training Loss: 1.0839
Epoch 15/50 | Training Loss: 0.6582
Epoch 20/50 | Training Loss: 0.4498
Epoch 25/50 | Training Loss: 0.2824
Epoch 30/50 | Training Loss: 0.2406
Epoch 35/50 | Training Loss: 0.1579
Epoch 40/50 | Training Loss: 0.1095
Epoch 45/50 | Training Loss: 0.0677
Epoch 50/50 | Training Loss: 0.0393

=== Unseen Test Set Metrics (20% Holdout) ===
Accuracy : 99.82%
AUROC    : 100.00%
AUPR     : 99.98%
Macro-F1 : 88.70%
